In [ ]:
%%file train.py


import os
import math
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.ops import sigmoid_focal_loss


from tqdm import tqdm
from rich.console import Console
from rich.panel import Panel
import evaluate
import wandb
import timm

from collections import defaultdict
import csv

from accelerate import Accelerator
from accelerate.utils import set_seed

from nvidia.dali import fn, types, pipeline_def
from nvidia.dali.plugin.pytorch import DALIGenericIterator, LastBatchPolicy


import torch.nn.functional as F



gradient_accumulation_steps = 1
accelerator = Accelerator(gradient_accumulation_steps=gradient_accumulation_steps)

device_id = accelerator.local_process_index
num_shards = accelerator.num_processes
shard_id = accelerator.local_process_index

console = Console()


checkpoint_dir = "checkpoint"
log_file = "metrics_log.txt"
training_state_path = os.path.join(checkpoint_dir, "training_state.pt")
best_loss_model_path = "best_val_loss_model.pth"
best_accuracy_model_path = "best_val_accuracy_model.pth"



seed = 42
set_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False




train_dir_1 = "/content/dataset1/"
train_dir_2 = "/content/dataset2/"
val_dir = "/content/dataset3/"



def create_metrics():
    metrics = {"accuracy": evaluate.load("accuracy"), "precision": evaluate.load("precision"), "recall": evaluate.load("recall"), "f1": evaluate.load("f1"),}
    return metrics


def update_metrics(metrics, preds, labels):
    for metric in metrics.values():
        metric.add_batch(predictions=preds, references=labels)






def compute_metrics(metrics):
    results = {}
    results["accuracy"] = metrics["accuracy"].compute()["accuracy"]
    results["precision"] = metrics["precision"].compute(average="macro")["precision"]
    results["recall"] = metrics["recall"].compute(average="macro")["recall"]
    results["f1"] = metrics["f1"].compute(average="macro")["f1"]
    return results





class DDDSequenceIndex():

    def __init__(self, root_dir, time_window_in_seconds, frames_per_window, window_stride_by_seconds):
        self.root_dir = root_dir
        self.time_window_in_seconds = time_window_in_seconds
        self.frames_per_window = frames_per_window
        self.window_stride_by_seconds = window_stride_by_seconds

        self.class_counts = {}
        self.sequence_frame_paths = []
        self.sequence_labels = []

        self.sequence_group_keys = []
        self.group_to_id = {}
        self.id_to_group = {}

        self._build_sequences()

    def _build_sequences(self):

        for subject_entry in os.scandir(self.root_dir):
            subject_name = subject_entry.name

            for scenario_entry in os.scandir(subject_entry.path):
                scenario_name = scenario_entry.name
                video_fps = 15 if "night" in scenario_name.lower() else 30

                for video_type_entry in os.scandir(scenario_entry.path):
                    self._process_windows_in_video(
                        video_path=video_type_entry.path,
                        video_fps=video_fps,
                        subject_name=subject_name,
                        scenario_name=scenario_name,
                    )

    def _process_windows_in_video(self, video_path, video_fps, subject_name, scenario_name):

        video_frame_list = sorted(entry.name for entry in os.scandir(video_path))
        total_frames_in_video = len(video_frame_list)
        available_frames_per_window = round(self.time_window_in_seconds * video_fps)

        stride_between_windows_in_frames = max(1, round(self.window_stride_by_seconds * video_fps))
        total_windows_in_video = (total_frames_in_video - available_frames_per_window) // stride_between_windows_in_frames + 1
        frames_space_in_window = (available_frames_per_window - 1) / (self.frames_per_window - 1)

        selected_frames_indices_in_window = [
            round(frame_position * frames_space_in_window)
            for frame_position in range(self.frames_per_window)
        ]

        last_frame_offset = selected_frames_indices_in_window[-1]

        frame_paths = [os.path.join(video_path, name) for name in video_frame_list]
        frame_labels = [int(name.rsplit("_", 1)[-1].split(".")[0]) for name in video_frame_list]

        append_paths = self.sequence_frame_paths.append
        append_label = self.sequence_labels.append
        append_group = self.sequence_group_keys.append
        class_counts = self.class_counts

        group_key = f"{subject_name} | {scenario_name}"   # NEW

        if group_key not in self.group_to_id:
            group_id = len(self.group_to_id)
            self.group_to_id[group_key] = group_id
            self.id_to_group[group_id] = group_key

        for window_index in range(total_windows_in_video):
            window_start_frame = window_index * stride_between_windows_in_frames

            last_frame_index = window_start_frame + last_frame_offset
            if last_frame_index >= total_frames_in_video:
                break

            frame_indices = [window_start_frame + frame_offset for frame_offset in selected_frames_indices_in_window]

            for frame_index in range(1, len(frame_indices)):
                if frame_indices[frame_index] <= frame_indices[frame_index - 1]:
                    frame_indices[frame_index] = frame_indices[frame_index - 1] + 1

            if frame_indices[-1] >= total_frames_in_video:
                break

            paths = [frame_paths[frame_index] for frame_index in frame_indices]
            label = frame_labels[frame_indices[-1]]

            class_counts[label] = class_counts.get(label, 0) + 1

            append_paths(paths)
            append_label(label)
            append_group(group_key)






class WeightedSequenceBatchSource():

    def __init__(self, sequence_frame_paths, sequence_labels, batch_size,
        frames_per_window, shuffle=True, seed=42, shard_id=0, num_shards=1, drop_last=True):

        self.sequence_frame_paths = sequence_frame_paths
        self.batch_size = batch_size
        self.frames_per_window = frames_per_window
        self.shuffle = shuffle
        self.seed = seed
        self.shard_id = shard_id
        self.num_shards = num_shards
        self.drop_last = drop_last
        self.rng = random.Random(seed + shard_id)

        self.sequence_labels = [self._normalize_label(label) for label in sequence_labels]

        dataset_indices = list(range(len(self.sequence_labels)))
        self.shard_indices = dataset_indices[self.shard_id::self.num_shards]

        shard_labels = [self.sequence_labels[i] for i in self.shard_indices]
        labels_tensor = torch.tensor(shard_labels, dtype=torch.long)

        num_classes = int(labels_tensor.max().item()) + 1
        class_counts = torch.bincount(labels_tensor, minlength=num_classes).float()

        class_weights = torch.zeros_like(class_counts)
        class_weights[class_counts > 0] = 1.0 / class_counts[class_counts > 0]

        sample_weights = class_weights[labels_tensor]
        self.sample_weights = sample_weights.tolist()

        self.sequence_frame_paths_encoded = [
            [np.frombuffer(path.encode("utf-8"), dtype=np.uint8) for path in seq_paths]
            for seq_paths in self.sequence_frame_paths
        ]

        self.sequence_repeated_labels = [
            [np.array([label], dtype=np.float32) for _ in range(self.frames_per_window)]
            for label in self.sequence_labels
        ]

        self.num_samples = len(self.shard_indices)
        self.cursor = 0
        self.sampled_indices = []
        self.reset()



    def _normalize_label(self, label):
        if isinstance(label, torch.Tensor):
            label = label.item()
        elif isinstance(label, np.ndarray):
            label = label.item()
        elif isinstance(label, (list, tuple)):
            label = label[0]

        return int(label)



    def reset(self):

        if self.shuffle:
            self.sampled_indices = self.rng.choices( self.shard_indices, weights=self.sample_weights, k=self.num_samples)
        else:
            self.sampled_indices = list(self.shard_indices)

        self.cursor = 0


    def __len__(self):
        if self.drop_last:
            return len(self.sampled_indices) // self.batch_size
        return math.ceil(len(self.sampled_indices) / self.batch_size)


    def __call__(self):
        if self.cursor >= len(self.sampled_indices):
            raise StopIteration

        batch_indices = self.sampled_indices[self.cursor:self.cursor + self.batch_size]

        if self.drop_last and len(batch_indices) < self.batch_size:
            raise StopIteration

        self.cursor += len(batch_indices)

        flat_paths = [
            path
            for idx in batch_indices
            for path in self.sequence_frame_paths_encoded[idx]
        ]

        flat_labels = [
            label
            for idx in batch_indices
            for label in self.sequence_repeated_labels[idx]
        ]

        return flat_paths, flat_labels




class SequentialBatchSource():

    def __init__(
        self, sequence_frame_paths, sequence_labels, sequence_group_keys,
        group_to_id, batch_size, frames_per_window, drop_last=True, shard_id=0, num_shards=1
    ):
        self.sequence_frame_paths = sequence_frame_paths
        self.sequence_labels = sequence_labels
        self.sequence_group_keys = sequence_group_keys
        self.group_to_id = group_to_id
        self.batch_size = batch_size
        self.frames_per_window = frames_per_window
        self.drop_last = drop_last
        self.shard_id = shard_id
        self.num_shards = num_shards

        dataset_window_indices = list(range(len(self.sequence_labels)))
        self.shard_window_indices = dataset_window_indices[self.shard_id::self.num_shards]

        if self.drop_last:
            usable = (len(self.shard_window_indices) // self.batch_size) * self.batch_size
            self.shard_window_indices = self.shard_window_indices[:usable]
            self.num_batches = len(self.shard_window_indices) // self.batch_size
        else:
            self.num_batches = math.ceil(len(self.shard_window_indices) / self.batch_size)

        self.sequence_frame_paths_encoded = [
            [np.frombuffer(path.encode("utf-8"), dtype=np.uint8) for path in seq_paths]
            for seq_paths in self.sequence_frame_paths
        ]

        self.sequence_repeated_labels = [
            [np.array([label], dtype=np.float32) for _ in range(self.frames_per_window)]
            for label in self.sequence_labels
        ]

        # NEW
        self.sequence_repeated_group_ids = [
            [np.array([self.group_to_id[group_key]], dtype=np.int32) for _ in range(self.frames_per_window)]
            for group_key in self.sequence_group_keys
        ]

        self.cursor = 0

    def reset(self):
        self.cursor = 0

    def __len__(self):
        return self.num_batches

    def __call__(self):
        if self.cursor >= len(self.shard_window_indices):
            raise StopIteration

        batch_indices = self.shard_window_indices[self.cursor:self.cursor + self.batch_size]
        self.cursor += self.batch_size

        if len(batch_indices) < self.batch_size and self.drop_last:
            raise StopIteration

        flat_paths = [path for idx in batch_indices for path in self.sequence_frame_paths_encoded[idx]]
        flat_labels = [label for idx in batch_indices for label in self.sequence_repeated_labels[idx]]
        flat_group_ids = [gid for idx in batch_indices for gid in self.sequence_repeated_group_ids[idx]]  # NEW

        return flat_paths, flat_labels, flat_group_ids








@pipeline_def
def train_sequence_pipeline(batch_source, image_height, image_width, frames_per_window):

    filepaths, labels = fn.external_source(
        source=batch_source,
        num_outputs=2,
        batch=True,
        parallel=False
    )

    encoded = fn.io.file.read(filepaths)

    images = fn.experimental.decoders.image(
        encoded,
        device="mixed",
        output_type=types.GRAY
    )

    images = fn.resize(
        images,
        resize_x=image_width,
        resize_y=image_height
    )

    brightness = fn.random.uniform(range=(0.9, 1.1))
    contrast = fn.random.uniform(range=(0.9, 1.1))

    images = fn.brightness_contrast(
        images,
        brightness=brightness,
        contrast=contrast
    )

    images = fn.crop_mirror_normalize(
        images,
        dtype=types.FLOAT,
        output_layout="CHW",
        mean=[127.5],
        std=[127.5],
    )


    labels = fn.cast(labels, dtype=types.FLOAT)
    return images, labels








@pipeline_def
def val_sequence_pipeline(batch_source, image_height, image_width, frames_per_window):
    filepaths, labels, group_ids = fn.external_source(
        source=batch_source,
        num_outputs=3,
        batch=True,
        parallel=False
    )

    encoded = fn.io.file.read(filepaths)
    images = fn.experimental.decoders.image(
        encoded,
        device="mixed",
        output_type=types.GRAY
    )

    images = fn.resize(images, resize_x=image_width, resize_y=image_height)

    images = fn.crop_mirror_normalize(
        images,
        dtype=types.FLOAT,
        output_layout="CHW",
        mean=[127.5],
        std=[127.5],
    )

    labels = fn.cast(labels, dtype=types.FLOAT)
    group_ids = fn.cast(group_ids, dtype=types.INT32)   # NEW
    return images, labels, group_ids





class DALISequenceLoader():

    def __init__(self, iterator, batch_source, frames_per_window):
        self.iterator = iterator
        self.batch_source = batch_source
        self.frames_per_window = frames_per_window

    def __iter__(self):
        return iter(self.iterator)

    def __len__(self):
        return len(self.batch_source)

    def reset(self):
        self.batch_source.reset()
        self.iterator.reset()






def save_group_validation_stats(group_stats, id_to_group, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "group_id",
            "group_name",
            "total_samples",
            "correct_samples",
            "correct_percentage",
            "predicted_class_0_count",
            "predicted_class_0_percentage",
            "actual_class_0_count",
            "actual_class_0_percentage",
        ])

        for group_id in sorted(group_stats.keys()):
            stats = group_stats[group_id]
            total = stats["total"]

            correct_pct = 100.0 * stats["correct"] / total if total > 0 else 0.0
            pred_class_0_pct = 100.0 * stats["pred_class_0"] / total if total > 0 else 0.0
            actual_class_0_pct = 100.0 * stats["actual_class_0"] / total if total > 0 else 0.0

            writer.writerow([
                group_id,
                id_to_group[group_id],
                total,
                stats["correct"],
                f"{correct_pct:.2f}",
                stats["pred_class_0"],
                f"{pred_class_0_pct:.2f}",
                stats["actual_class_0"],
                f"{actual_class_0_pct:.2f}",
            ])






def create_dali_dataloaders(accelerator, train_dirs, val_dir, batch_size, time_window_in_seconds,
    frames_per_window, window_stride_by_seconds, prefetch_queue_depth, image_height, image_width):

    device_id = accelerator.local_process_index
    shard_id = accelerator.local_process_index
    num_shards = accelerator.num_processes

    train_sequence_frame_paths = []
    train_sequence_labels = []
    combined_class_counts = {}

    for train_dir in train_dirs:
        train_index = DDDSequenceIndex(
            root_dir=train_dir,
            time_window_in_seconds=time_window_in_seconds,
            frames_per_window=frames_per_window,
            window_stride_by_seconds=window_stride_by_seconds,
        )

        train_sequence_frame_paths.extend(train_index.sequence_frame_paths)
        train_sequence_labels.extend(train_index.sequence_labels)

        for class_label, count in train_index.class_counts.items():
            combined_class_counts[class_label] = (
                combined_class_counts.get(class_label, 0) + count
            )

    val_index = DDDSequenceIndex(
        root_dir=val_dir,
        time_window_in_seconds=time_window_in_seconds,
        frames_per_window=frames_per_window,
        window_stride_by_seconds=window_stride_by_seconds,
    )

    accelerator.print("Combined training sequences:")
    for class_label in sorted(combined_class_counts):
        accelerator.print(f"Class {class_label} sequences: {combined_class_counts[class_label]}")
    accelerator.print(f"Total sequences: {len(train_sequence_frame_paths)}\n")

    accelerator.print("Validation sequences:")
    for class_label in sorted(val_index.class_counts):
        accelerator.print(f"Class {class_label} sequences: {val_index.class_counts[class_label]}")
    accelerator.print(f"Total sequences: {len(val_index.sequence_frame_paths)}\n")

    train_source = WeightedSequenceBatchSource(
        sequence_frame_paths=train_sequence_frame_paths,
        sequence_labels=train_sequence_labels,
        batch_size=batch_size,
        frames_per_window=frames_per_window,
        shuffle=True,
        drop_last=True,
        seed=seed + shard_id,
        shard_id=shard_id,
        num_shards=num_shards,
    )


    val_source = SequentialBatchSource(
        sequence_frame_paths=val_index.sequence_frame_paths,
        sequence_labels=val_index.sequence_labels,
        sequence_group_keys=val_index.sequence_group_keys,
        group_to_id=val_index.group_to_id,
        batch_size=batch_size,
        frames_per_window=frames_per_window,
        drop_last=False,
        shard_id=shard_id,
        num_shards=num_shards,
    )


    train_pipe = train_sequence_pipeline(
        batch_source=train_source,
        image_height=image_height,
        image_width=image_width,
        frames_per_window=frames_per_window,
        batch_size=batch_size * frames_per_window,
        num_threads=4,
        device_id=device_id,
        seed=seed + shard_id,
        prefetch_queue_depth=prefetch_queue_depth,
    )

    val_pipe = val_sequence_pipeline(
        batch_source=val_source,
        image_height=image_height,
        image_width=image_width,
        frames_per_window=frames_per_window,
        batch_size=batch_size * frames_per_window,
        num_threads=4,
        device_id=device_id,
        seed=seed + shard_id,
        prefetch_queue_depth=prefetch_queue_depth,
    )

    train_pipe.build()
    val_pipe.build()

    train_iter = DALIGenericIterator(
        [train_pipe],
        output_map=["frames", "labels"],
        size=-1,
        auto_reset=False,
        prepare_first_batch=True,
    )

    val_iter = DALIGenericIterator(
        [val_pipe],
        output_map=["frames", "labels", "group_ids"],
        size=-1,
        auto_reset=False,
        prepare_first_batch=True,
    )

    train_loader = DALISequenceLoader(train_iter, train_source, frames_per_window)
    val_loader = DALISequenceLoader(val_iter, val_source, frames_per_window)

    return train_loader, val_loader, combined_class_counts, val_index.id_to_group






def setup_training(accelerator, checkpoint_dir, training_state_path, log_file, device):

    best_val_loss = float("inf")
    best_val_accuracy = 0.0
    start_epoch = 0

    if os.path.isdir(checkpoint_dir) and os.listdir(checkpoint_dir):
        accelerator.print("⏭️ Loading checkpoint...")
        accelerator.load_state(checkpoint_dir)

        if os.path.exists(training_state_path):

            training_state = torch.load(training_state_path, map_location=device)
            start_epoch = training_state.get("epoch", 0)
            best_val_loss = training_state.get("best_val_loss", float("inf"))
            best_val_accuracy = training_state.get("best_val_accuracy", 0.0)
            accelerator.print(f"Resuming from epoch {start_epoch}")

        else:
            accelerator.print("🎯 No training state file found, starting from scratch.")

    if accelerator.is_main_process and start_epoch == 0:
        with open(log_file, "w") as f:
            f.write("epoch,train_loss,train_acc,train_precision,train_recall,train_f1,val_loss,val_acc,val_precision,val_recall,val_f1\n")

    return start_epoch, best_val_loss, best_val_accuracy






def run_train_epoch(
    model, train_loader, optimizer, accelerator, criterion,
    frames_per_window, classification_type, epoch, epochs
):
    model.train()

    if accelerator.is_main_process:
        print(f"\nEpoch {epoch + 1}/{epochs}")

    train_metrics = create_metrics()
    train_loss = 0.0
    num_train_batches = 0

    train_bar = tqdm(train_loader, desc="Training", disable=not accelerator.is_main_process)

    for batch_idx, data in enumerate(train_bar):
        batch = data[0]
        frames = batch["frames"]
        labels = batch["labels"]

        frames, labels = prepare_sequence_batch(
            frames=frames,
            labels=labels,
            frames_per_window=frames_per_window,
            classification_type=classification_type,
        )

        with accelerator.accumulate(model):
            outputs = model(frames)
            labels = labels.to(outputs.device, non_blocking=True)

            if classification_type == "binary":
                outputs = outputs.view(-1)
                labels = labels.view(-1).float()
            else:
                labels = labels.view(-1).long()

            loss = criterion(outputs, labels)
            accelerator.backward(loss)

            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        gathered_loss = accelerator.gather_for_metrics(loss.detach().unsqueeze(0))
        train_loss += gathered_loss.mean().item()
        num_train_batches += 1

        preds = get_predictions(outputs.detach(), classification_type)

        if classification_type == "binary":
            labels_for_metrics = labels.long()
        else:
            labels_for_metrics = labels

        preds_g, labels_g = accelerator.gather_for_metrics((preds, labels_for_metrics))
        update_metrics(train_metrics, preds_g, labels_g)

    train_loader.reset()

    avg_train_loss = train_loss / max(num_train_batches, 1)
    train_results = compute_metrics(train_metrics)

    return avg_train_loss, train_results










def find_best_threshold_vectorized(all_probs, all_labels, threshold_candidates):
    thresholds = torch.tensor(threshold_candidates, device=all_probs.device, dtype=all_probs.dtype)
    labels = all_labels.to(all_probs.device).long()

    pred_matrix = (all_probs[:, None] >= thresholds[None, :]).long()
    labels_exp = labels[:, None]

    tp = ((pred_matrix == 1) & (labels_exp == 1)).sum(dim=0).float()
    fp = ((pred_matrix == 1) & (labels_exp == 0)).sum(dim=0).float()
    tn = ((pred_matrix == 0) & (labels_exp == 0)).sum(dim=0).float()
    fn = ((pred_matrix == 0) & (labels_exp == 1)).sum(dim=0).float()

    accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    best_idx = torch.argmax(accuracy)

    return {
        "best_threshold": thresholds[best_idx].item(),
        "best_accuracy": accuracy[best_idx].item(),
        "best_precision": precision[best_idx].item(),
        "best_recall": recall[best_idx].item(),
        "best_f1": f1[best_idx].item(),
    }






def run_validation(
    model, val_loader, accelerator, criterion, frames_per_window,
    classification_type, id_to_group=None, epoch=None, group_stats_dir=None):
    model.eval()

    val_loss = 0.0
    num_val_batches = 0
    val_metrics = create_metrics()

    all_probs = []
    all_labels = []

    group_stats = defaultdict(lambda: {
        "total": 0,
        "correct": 0,
        "pred_class_0": 0,
        "actual_class_0": 0,
    })

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc="Validation", disable=not accelerator.is_main_process)

        for batch_idx, data in enumerate(val_bar):
            batch = data[0]
            frames = batch["frames"]
            labels = batch["labels"]
            group_ids = batch["group_ids"]

            frames, labels = prepare_sequence_batch(
                frames=frames,
                labels=labels,
                frames_per_window=frames_per_window,
                classification_type=classification_type,
            )

            group_ids = group_ids.view(-1, frames_per_window)[:, 0].long().to(frames.device, non_blocking=True)

            outputs = model(frames)
            labels = labels.to(outputs.device, non_blocking=True)

            if classification_type == "binary":
                outputs = outputs.view(-1)
                labels = labels.view(-1).float()
            else:
                labels = labels.view(-1).long()

            loss = criterion(outputs, labels)

            gathered_loss = accelerator.gather_for_metrics(loss.detach().unsqueeze(0))
            val_loss += gathered_loss.mean().item()
            num_val_batches += 1

            if classification_type == "binary":
                probs = torch.sigmoid(outputs.detach())
                preds = (probs >= 0.5).long()
                labels_for_metrics = labels.long()

                probs_g, labels_prob_g = accelerator.gather_for_metrics((probs, labels_for_metrics))
                all_probs.append(probs_g)
                all_labels.append(labels_prob_g)
            else:
                preds = torch.argmax(outputs.detach(), dim=1)
                labels_for_metrics = labels

            preds_g, labels_g, group_ids_g = accelerator.gather_for_metrics(
                (preds, labels_for_metrics, group_ids)
            )

            update_metrics(val_metrics, preds_g, labels_g)

            for pred, label, group_id in zip(
                preds_g.cpu().tolist(),
                labels_g.cpu().tolist(),
                group_ids_g.cpu().tolist()
            ):
                group_stats[group_id]["total"] += 1
                group_stats[group_id]["correct"] += int(pred == label)
                group_stats[group_id]["pred_class_0"] += int(pred == 0)
                group_stats[group_id]["actual_class_0"] += int(label == 0)

    val_results = compute_metrics(val_metrics)
    avg_val_loss = val_loss / max(num_val_batches, 1)

    if classification_type == "binary" and len(all_probs) > 0:
        all_probs = torch.cat(all_probs, dim=0)
        all_labels = torch.cat(all_labels, dim=0)

        threshold_candidates = [0.4, 0.45, 0.47, 0.5, 0.52, 0.55, 0.6]

        best_results = find_best_threshold_vectorized(
            all_probs=all_probs,
            all_labels=all_labels,
            threshold_candidates=threshold_candidates,
        )

        val_results["best_threshold"] = best_results["best_threshold"]
        val_results["best_accuracy"] = best_results["best_accuracy"]
        val_results["best_precision"] = best_results["best_precision"]
        val_results["best_recall"] = best_results["best_recall"]
        val_results["best_f1"] = best_results["best_f1"]

    if accelerator.is_main_process and id_to_group is not None:
        print("\n" + "=" * 80)
        print("VALIDATION - PER PERSON / CONDITION STATS")
        print("=" * 80)

        for group_id in sorted(
            group_stats.keys(),
            key=lambda g: group_stats[g]["correct"] / max(group_stats[g]["total"], 1)
        ):
            stats = group_stats[group_id]
            total = stats["total"]

            correct_pct = 100.0 * stats["correct"] / total if total > 0 else 0.0
            pred_class_0_pct = 100.0 * stats["pred_class_0"] / total if total > 0 else 0.0
            actual_class_0_pct = 100.0 * stats["actual_class_0"] / total if total > 0 else 0.0

            print(
                f"{id_to_group[group_id]:<40} | "
                f"Total: {total:4d} | "
                f"Correct: {stats['correct']:4d} ({correct_pct:6.2f}%) | "
                f"Pred Class0: {stats['pred_class_0']:4d} ({pred_class_0_pct:6.2f}%) | "
                f"Actual Class0: {stats['actual_class_0']:4d} ({actual_class_0_pct:6.2f}%)"
            )

        print("=" * 80 + "\n")

        if group_stats_dir is not None and epoch is not None:
            save_group_validation_stats(
                group_stats=group_stats,
                id_to_group=id_to_group,
                output_path=os.path.join(group_stats_dir, f"val_group_stats_epoch_{epoch + 1}.csv")
            )

    return avg_val_loss, val_results



def log_epoch_results(
    epoch, avg_train_loss, train_results, avg_val_loss, val_results,
    log_file, accelerator, classification_type
):
    if accelerator.is_main_process:
        print(
            f"Epoch {epoch + 1} - Train Loss: {avg_train_loss:.4f} | "
            f"Acc: {train_results['accuracy']:.4f} | "
            f"Precision: {train_results['precision']:.4f} | "
            f"Recall: {train_results['recall']:.4f} | "
            f"F1: {train_results['f1']:.4f}"
        )

        if classification_type == "binary":
            print(
                f"Epoch {epoch + 1} - Val   Loss: {avg_val_loss:.4f} | "
                f"Acc@0.5: {val_results['accuracy']:.4f} | "
                f"Precision@0.5: {val_results['precision']:.4f} | "
                f"Recall@0.5: {val_results['recall']:.4f} | "
                f"F1@0.5: {val_results['f1']:.4f}"
            )

            if "best_threshold" in val_results:
                print(
                    f"Epoch {epoch + 1} - Val Best Threshold: {val_results['best_threshold']:.2f} | "
                    f"Best Acc: {val_results['best_accuracy']:.4f} | "
                    f"Best Precision: {val_results['best_precision']:.4f} | "
                    f"Best Recall: {val_results['best_recall']:.4f} | "
                    f"Best F1: {val_results['best_f1']:.4f}"
                )
        else:
            print(
                f"Epoch {epoch + 1} - Val   Loss: {avg_val_loss:.4f} | "
                f"Acc: {val_results['accuracy']:.4f} | "
                f"Precision: {val_results['precision']:.4f} | "
                f"Recall: {val_results['recall']:.4f} | "
                f"F1: {val_results['f1']:.4f}"
            )

        log_data = {
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "train_accuracy": train_results["accuracy"],
            "train_precision": train_results["precision"],
            "train_recall": train_results["recall"],
            "train_f1": train_results["f1"],
            "val_loss": avg_val_loss,
            "val_accuracy": val_results["accuracy"],
            "val_precision": val_results["precision"],
            "val_recall": val_results["recall"],
            "val_f1": val_results["f1"],
        }

        if classification_type == "binary" and "best_threshold" in val_results:
            log_data["val_best_threshold"] = val_results["best_threshold"]
            log_data["val_best_accuracy"] = val_results["best_accuracy"]
            log_data["val_best_precision"] = val_results["best_precision"]
            log_data["val_best_recall"] = val_results["best_recall"]
            log_data["val_best_f1"] = val_results["best_f1"]

        wandb.log(log_data)

        with open(log_file, "a") as f:
            if classification_type == "binary" and "best_threshold" in val_results:
                f.write(
                    f"{epoch + 1},"
                    f"{avg_train_loss:.4f},{train_results['accuracy']:.4f},{train_results['precision']:.4f},{train_results['recall']:.4f},{train_results['f1']:.4f},"
                    f"{avg_val_loss:.4f},{val_results['accuracy']:.4f},{val_results['precision']:.4f},{val_results['recall']:.4f},{val_results['f1']:.4f},"
                    f"{val_results['best_threshold']:.2f},{val_results['best_accuracy']:.4f},{val_results['best_precision']:.4f},{val_results['best_recall']:.4f},{val_results['best_f1']:.4f}\n"
                )
            else:
                f.write(
                    f"{epoch + 1},"
                    f"{avg_train_loss:.4f},{train_results['accuracy']:.4f},{train_results['precision']:.4f},{train_results['recall']:.4f},{train_results['f1']:.4f},"
                    f"{avg_val_loss:.4f},{val_results['accuracy']:.4f},{val_results['precision']:.4f},{val_results['recall']:.4f},{val_results['f1']:.4f}\n"
                )






def save_best_models(model, accelerator, epoch, avg_val_loss, val_results,
    best_val_loss, best_val_accuracy, best_loss_model_path, best_accuracy_model_path):

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss

        if accelerator.is_main_process:
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": accelerator.get_state_dict(model),
                    "best_val_loss": best_val_loss,
                    "best_val_accuracy": best_val_accuracy,
                    "best_threshold": val_results.get("best_threshold", 0.5),
                    "best_precision": val_results.get("best_precision", val_results.get("precision")),
                    "best_recall": val_results.get("best_recall", val_results.get("recall")),
                    "best_f1": val_results.get("best_f1", val_results.get("f1")),
                },
                best_loss_model_path,
            )

            artifact = wandb.Artifact(
                f"{experiment_no}-best-loss-model",
                type="model"
            )
            artifact.add_file(
                best_loss_model_path,
                name=f"{experiment_no}/best_loss_model/model.pth"
            )
            wandb.log_artifact(artifact)

            print("⭐ Best val loss model updated!")

    # 2) Best model by searched threshold accuracy
    metric_for_best_acc = val_results.get("best_accuracy", val_results["accuracy"])

    if metric_for_best_acc > best_val_accuracy:
        best_val_accuracy = metric_for_best_acc

        if accelerator.is_main_process:
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": accelerator.get_state_dict(model),
                    "best_val_loss": best_val_loss,
                    "best_val_accuracy": best_val_accuracy,
                    "best_threshold": val_results.get("best_threshold", 0.5),
                    "best_precision": val_results.get("best_precision", val_results.get("precision")),
                    "best_recall": val_results.get("best_recall", val_results.get("recall")),
                    "best_f1": val_results.get("best_f1", val_results.get("f1")),
                },
                best_accuracy_model_path,
            )

            artifact = wandb.Artifact(
                f"{experiment_no}-best-acc-model",
                type="model"
            )
            artifact.add_file(
                best_accuracy_model_path,
                name=f"{experiment_no}/best_acc_model/model.pth"
            )
            wandb.log_artifact(artifact)

            print(
                f"🔥 Best val accuracy model updated! "
                f"(threshold={val_results.get('best_threshold', 0.5):.2f}, "
                f"acc={metric_for_best_acc:.4f})"
            )

        export_branch_features(
            model=model,
            loader=val_loader,
            accelerator=accelerator,
            frames_per_window=config["frames_per_window"],
            classification_type=config["classification_type"],
            output_path=f"best_{config['model_name']}_{task}_features.npz",
        )

        val_loader.reset()

    return best_val_loss, best_val_accuracy








def get_predictions(outputs, classification_type, threshold=0.5):
    if classification_type == "binary":
        outputs = outputs.view(-1)
        preds = (torch.sigmoid(outputs) >= threshold).long()
        return preds

    elif classification_type == "multiclass":
        preds = torch.argmax(outputs, dim=1)
        return preds

    else:
        raise ValueError("classification_type must be 'binary' or 'multiclass'")





def prepare_sequence_batch(frames, labels, frames_per_window, classification_type):
    flat_batch_size = frames.shape[0]
    real_batch_size = flat_batch_size // frames_per_window

    frames = frames.view(real_batch_size, frames_per_window, *frames.shape[1:])
    labels = labels.view(real_batch_size, frames_per_window, -1)[:, -1, 0]

    if classification_type == "binary":
        labels = labels.view(-1).float()
    elif classification_type == "multiclass":
        labels = labels.view(-1).long()
    else:
        raise ValueError("classification_type must be 'binary' or 'multiclass'")

    return frames, labels




def create_optimizer(model, config):
    temporal_lr = config["temporal_lr"]

    def trainable_params(module):
        return [p for p in module.parameters() if p.requires_grad]

    param_groups = [
        {"params": trainable_params(model.encoder.enhancer), "lr": config["encoder_head_lr"]},
        {"params": trainable_params(model.encoder.backbone), "lr": config["encoder_backbone_lr"]},
        {"params": trainable_params(model.encoder.fc),       "lr": config["encoder_head_lr"]},
    ]

    if hasattr(model, "lstm"):
        param_groups.append({"params": trainable_params(model.lstm), "lr": temporal_lr})

    if hasattr(model, "tcn"):
        param_groups.append({"params": trainable_params(model.tcn), "lr": temporal_lr})

    if hasattr(model, "transformer"):
        param_groups.append({"params": trainable_params(model.transformer), "lr": temporal_lr})

    if hasattr(model, "pos_embedding") and model.pos_embedding.requires_grad:
        param_groups.append({"params": [model.pos_embedding], "lr": temporal_lr})

    param_groups.append({"params": trainable_params(model.classifier), "lr": temporal_lr})
    param_groups = [group for group in param_groups if len(group["params"]) > 0]
    return torch.optim.AdamW(param_groups, weight_decay=config["weight_decay"])




def get_criterion(loss_type, device, class_counts=None, config=None):
    loss_type = loss_type.lower()

    if loss_type == "focal_binary":
        alpha = -1
        gamma = 2.0

        def criterion(outputs, labels):
            return sigmoid_focal_loss(
                inputs=outputs,
                targets=labels.float(),
                alpha=alpha,
                gamma=gamma,
                reduction="mean"
            )

        return criterion

    elif loss_type == "bce":
        positive_weight_value = 1.0
        if config is not None:
            positive_weight_value = config.get("pos_weight", 1.0)

        positive_weight = torch.tensor(
            [positive_weight_value],
            dtype=torch.float32,
            device=device
        )

        return torch.nn.BCEWithLogitsLoss(pos_weight=positive_weight)

    elif loss_type == "ce":
        weight = None

        if class_counts is not None:
            if isinstance(class_counts, dict):
                max_class = max(class_counts.keys())
                counts = [class_counts.get(i, 0) for i in range(max_class + 1)]
            else:
                counts = class_counts

            class_counts_tensor = torch.tensor(counts, dtype=torch.float32, device=device)
            weight = torch.zeros_like(class_counts_tensor)
            valid = class_counts_tensor > 0
            weight[valid] = class_counts_tensor.sum() / class_counts_tensor[valid]
            weight = weight / weight.sum() * len(class_counts_tensor)

        return torch.nn.CrossEntropyLoss(weight=weight)

    else:
        raise ValueError(f"Unsupported loss_type: {loss_type}")





class EnhancementBlock(nn.Module):

    def __init__(self, channels=1, hidden_channels=8):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(channels, hidden_channels, kernel_size=3, padding=1),
            nn.GroupNorm(num_groups=4, num_channels=hidden_channels),
            nn.GELU(),
            nn.Conv2d(hidden_channels, channels, kernel_size=3, padding=1)
        )

    def forward(self, x):
        enhanced = self.block(x)
        return x + 0.1 * enhanced





class CNNEncoder(nn.Module):

    def __init__(self, encoder_backbone_name, encoder_output_dim, encoder_dropout):

        super().__init__()
        self.enhancer = EnhancementBlock(channels=1, hidden_channels=8)

        self.backbone = timm.create_model(
            encoder_backbone_name,
            pretrained=True,
            in_chans=1,
            num_classes=0,
            global_pool=""
        )

        for param in self.backbone.parameters():
            param.requires_grad = True

        if hasattr(self.backbone, "conv_head"):
            for param in self.backbone.conv_head.parameters():
                param.requires_grad = False

        backbone_output_dim = self.backbone.num_features

        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )

        self.fc = nn.Sequential(
            nn.Linear(backbone_output_dim, encoder_output_dim),
            nn.LayerNorm(encoder_output_dim),
            nn.GELU(),
            nn.Dropout(encoder_dropout),
        )

    def forward(self, x):
        x = self.enhancer(x)
        x = self.backbone.forward_features(x)

        if x.ndim == 4:
            x = self.pool(x)
        elif x.ndim == 2:
            pass
        else:
            raise ValueError(f"Unexpected feature shape: {x.shape}")

        x = self.fc(x)
        return x




class LSTM(nn.Module):

    def __init__(self, encoder_backbone_name, encoder_output_dim, encoder_dropout, lstm_hidden_dim, num_classes, lstm_num_layers,
                  temporal_dropout, proj_size=0):
        super().__init__()
        self.num_classes = num_classes
        self.is_binary = num_classes == 1

        self.encoder = CNNEncoder(
            encoder_backbone_name=encoder_backbone_name,
            encoder_output_dim=encoder_output_dim,
            encoder_dropout=encoder_dropout,
        )

        self.lstm = nn.LSTM(
            input_size=encoder_output_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=lstm_num_layers,
            batch_first=True,
            dropout=temporal_dropout if lstm_num_layers > 1 else 0.0,
            proj_size=proj_size,
        )

        final_dim = proj_size if proj_size > 0 else lstm_hidden_dim
        self.classifier = nn.Linear(final_dim * 2, num_classes)

    def forward_features(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        features = self.encoder(x)
        features = features.view(B, T, -1)

        lstm_out, _ = self.lstm(features)
        last_timestep = lstm_out[:, -2:, :]      # [B, 2, H]
        last_timestep = last_timestep.reshape(B, -1)   # [B, 2H]
        return last_timestep

    def forward(self, x):
        last_timestep = self.forward_features(x)
        logits = self.classifier(last_timestep)

        if self.is_binary:
            return logits.squeeze(1)
        return logits





class CausalTemporalConvBlock(nn.Module):

    def __init__(self, in_channels, out_channels, dilation, temporal_dropout, num_groups=8, kernel_size=3):
        super().__init__()
        self.padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=self.padding, dilation=dilation)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=self.padding, dilation=dilation)
        self.conv3 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=self.padding, dilation=dilation)

        self.norm1 = nn.GroupNorm(num_groups=num_groups, num_channels=out_channels)
        self.norm2 = nn.GroupNorm(num_groups=num_groups, num_channels=out_channels)
        self.norm3 = nn.GroupNorm(num_groups=num_groups, num_channels=out_channels)

        self.activation = nn.GELU()
        self.dropout = nn.Dropout(temporal_dropout)

        self.residual = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):
        residual = self.residual(x)

        out = self.conv1(x)
        if self.padding > 0:
            out = out[:, :, :-self.padding]
        out = self.norm1(out)
        out = self.activation(out)
        out = self.dropout(out)

        out = self.conv2(out)
        if self.padding > 0:
            out = out[:, :, :-self.padding]
        out = self.norm2(out)
        out = self.activation(out)
        out = self.dropout(out)

        out = self.conv3(out)
        if self.padding > 0:
            out = out[:, :, :-self.padding]
        out = self.norm3(out)
        out = self.activation(out)
        out = self.dropout(out)

        return self.activation(out + residual)






class TemporalConv(nn.Module):
    def __init__(self, encoder_backbone_name,
        encoder_output_dim, encoder_dropout,  temporal_dropout, tcn_hidden_dim, num_classes,):

        super().__init__()
        self.num_classes = num_classes
        self.is_binary = num_classes == 1

        self.encoder = CNNEncoder(
            encoder_backbone_name=encoder_backbone_name,
            encoder_output_dim=encoder_output_dim,
            encoder_dropout=encoder_dropout,
        )

        # keep your existing TCN blocks here
        self.tcn = nn.Sequential(
            *[
                CausalTemporalConvBlock(
                    in_channels=encoder_output_dim if i == 0 else tcn_hidden_dim[i - 1],
                    out_channels=tcn_hidden_dim[i],
                    dilation=2 ** i,
                    dropout=temporal_dropout,
                )
                for i in range(len(tcn_hidden_dim))
            ]
        )

        self.classifier = nn.Linear(tcn_hidden_dim[-1], num_classes)

    def forward_features(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        features = self.encoder(x)
        features = features.view(B, T, -1)          # (B, T, F)

        features = features.transpose(1, 2)         # (B, F, T)
        tcn_out = self.tcn(features)                # (B, hidden, T)
        last_timestep = tcn_out[:, :, -1]           # (B, hidden)
        return last_timestep

    def forward(self, x):
        last_timestep = self.forward_features(x)
        logits = self.classifier(last_timestep)

        if self.is_binary:
            return logits.squeeze(1)
        return logits






class TransformerEncoder(nn.Module):
    def __init__(self, encoder_backbone_name, encoder_output_dim, encoder_dropout, temporal_dropout,
                 num_heads, num_layers, dim_feedforward, max_len, num_classes):
        super().__init__()
        self.num_classes = num_classes
        self.is_binary = num_classes == 1

        self.encoder = CNNEncoder(
            encoder_backbone_name=encoder_backbone_name,
            encoder_output_dim=encoder_output_dim,
            encoder_dropout=encoder_dropout,
        )

        self.pos_embedding = nn.Parameter(torch.randn(1, max_len, encoder_output_dim))

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=encoder_output_dim,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=temporal_dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(transformer_layer, num_layers=num_layers)
        self.classifier = nn.Linear(encoder_output_dim, num_classes)

    def _generate_causal_mask(self, seq_len, device, dtype):
        mask = torch.triu(
            torch.full((seq_len, seq_len), float("-inf"), device=device, dtype=dtype),
            diagonal=1
        )
        return mask

    def forward_features(self, x):
        B, T, C, H, W = x.shape

        if T > self.pos_embedding.size(1):
            raise ValueError(f"Sequence length T={T} exceeds max_len={self.pos_embedding.size(1)}")

        x = x.view(B * T, C, H, W)
        features = self.encoder(x)
        features = features.view(B, T, -1)
        features = features + self.pos_embedding[:, :T, :]

        causal_mask = self._generate_causal_mask(
            seq_len=T,
            device=features.device,
            dtype=features.dtype
        )

        transformed = self.transformer(features, mask=causal_mask)
        last_timestep = transformed[:, -1, :]
        return last_timestep

    def forward(self, x):
        last_timestep = self.forward_features(x)
        logits = self.classifier(last_timestep)

        if self.is_binary:
            return logits.squeeze(1)
        return logits



def export_branch_features(
    model,
    loader,
    accelerator,
    frames_per_window,
    classification_type,
    output_path,
):
    model.eval()

    all_features = []
    all_probs = []
    all_labels = []
    all_ids = []

    with torch.no_grad():
        for data in loader:
            batch = data[0]
            frames = batch["frames"]
            labels = batch["labels"]

            frames, labels = prepare_sequence_batch(
                frames=frames,
                labels=labels,
                frames_per_window=frames_per_window,
                classification_type=classification_type,
            )

            features = model.forward_features(frames)   # <- latent embedding
            outputs = model.classifier(features)        # <- logits from embedding

            if classification_type == "binary":
                outputs = outputs.view(-1)
                probs = torch.sigmoid(outputs)
                labels = labels.view(-1).long()
            else:
                probs = F.softmax(outputs, dim=1)
                labels = labels.view(-1).long()

            features_g = accelerator.gather_for_metrics(features)
            probs_g = accelerator.gather_for_metrics(probs)
            labels_g = accelerator.gather_for_metrics(labels)

            features_np = features_g.cpu().numpy()
            probs_np = probs_g.cpu().numpy()
            labels_np = labels_g.cpu().numpy()

            batch_size = len(labels_np)
            ids = np.arange(len(all_ids), len(all_ids) + batch_size)

            all_features.append(features_np)
            all_probs.append(probs_np)
            all_labels.append(labels_np)
            all_ids.extend(ids.tolist())



    if accelerator.is_main_process:
        all_features = np.concatenate(all_features, axis=0)
        all_probs = np.concatenate(all_probs, axis=0)
        all_labels = np.concatenate(all_labels, axis=0)
        all_ids = np.array(all_ids)
        
        np.savez(
            output_path,
            ids=all_ids,
            features=all_features,
            probs=all_probs,
            labels=all_labels,
        )
        print(f"✅ Saved branch embeddings → {output_path}")





def build_model(config):
    if config["model_name"] == "lstm":
        model = LSTM(
            encoder_backbone_name=config["encoder_backbone_name"],
            encoder_output_dim=config["encoder_output_dim"],
            encoder_dropout=config["encoder_dropout"],
            lstm_hidden_dim=config["lstm_hidden_dim"],
            num_classes=config["num_classes"],
            lstm_num_layers=config["lstm_num_layers"],
            temporal_dropout=config["temporal_dropout"],
            proj_size=config.get("proj_size", 0)
        )

    elif config["model_name"] == "temporal_conv":
        model = TemporalConv(
            encoder_backbone_name=config["encoder_backbone_name"],
            encoder_output_dim=config["encoder_output_dim"],
            encoder_dropout=config["encoder_dropout"],
            temporal_dropout=config["temporal_dropout"],
            tcn_hidden_dim=config["tcn_hidden_dim"],
            num_classes=config["num_classes"],
        )

    elif config["model_name"] == "transformer_encoder":
        model = TransformerEncoder(
            encoder_backbone_name=config["encoder_backbone_name"],
            encoder_output_dim=config["encoder_output_dim"],
            encoder_dropout=config["encoder_dropout"],
            temporal_dropout=config["temporal_dropout"],
            num_heads=config["transformer_num_heads"],
            num_layers=config["transformer_num_layers"],
            dim_feedforward=config["transformer_hidden_dim"],
            max_len=config["frames_per_window"],
            num_classes=config["num_classes"],
        )

    else:
        raise ValueError(f"Unknown model_name: {config['model_name']}")

    return model




def get_config():
    model_name = "lstm"

    config = {
        "experiment_no": 72,

        "epochs": 180,
        "batch_size": 64,
        "gradient_accumulation_steps": 1,
        "train_batch_strategy": "weighted_sampling_with_replacement",

        "image_height": 120,
        "image_width": 120,

        "prefetch_queue_depth": 256,
        "encoder_backbone_name": "mobilenetv3_small_100",

        "time_window_in_seconds": 3,
        "frames_per_window": 16,
        "window_stride_by_seconds": 2,

        "encoder_output_dim": 288,
        "encoder_dropout": 0.2,
        "temporal_dropout": 0.2,

        "optimizer": "AdamW",
        "encoder_backbone_lr": 0.0001,
        "encoder_head_lr": 0.001,
        "temporal_lr": 0.001,
        "weight_decay": 0.0001,

        "classification_type": classification_type,
        "loss_type": "ce",
        "num_classes": 3,
    }

    if model_name == "lstm":
        config.update({
            "model_name": "lstm",
            "lstm_hidden_dim": 288,
            "lstm_num_layers": 2,
            "projection_head": False,
            "proj_size": 0,
        })

    elif model_name == "temporal_conv":
        config.update({
            "model_name": "temporal_conv",
            "tcn_hidden_dim": [512, 512, 512],
        })

    elif model_name == "transformer_encoder":
        config.update({
            "model_name": "transformer_encoder",
            "transformer_hidden_dim": 512,
            "transformer_num_layers": 4,
            "transformer_num_heads": 8,
        })

    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    return config




task = "head" # "head" "mouth" "righteye"
task_type = {"righteye": "binary", "mouth": "multiclass", "head": "multiclass",}
classification_type = task_type[task]

config = get_config()



experiment_no = config["experiment_no"]
if accelerator.is_main_process:
    console.print(Panel.fit(f"[bold yellow]Running Experiment:[/bold yellow]\n[bold green]{experiment_no}[/bold green]",border_style="bright_blue"))


train_loader, val_loader, class_counts, val_id_to_group = create_dali_dataloaders(
    accelerator=accelerator,
    train_dirs=[train_dir_1, train_dir_2],
    val_dir=val_dir,
    batch_size=config["batch_size"],
    time_window_in_seconds=config["time_window_in_seconds"],
    frames_per_window=config["frames_per_window"],
    window_stride_by_seconds=config["window_stride_by_seconds"],
    image_height=config["image_height"],
    image_width=config["image_width"],
    prefetch_queue_depth=config["prefetch_queue_depth"],
)



model = build_model(config)
optimizer = create_optimizer(model, config)


model, optimizer  = accelerator.prepare(model, optimizer)
device = accelerator.device


criterion = get_criterion(
    loss_type=config["loss_type"],
    device=accelerator.device,
    class_counts=class_counts if config["classification_type"] == "multiclass" else None,
    config=config,
)


start_epoch, best_val_loss, best_val_accuracy = setup_training(
    accelerator=accelerator,
    checkpoint_dir=checkpoint_dir,
    training_state_path=training_state_path,
    log_file=log_file,
    device=device
)

if accelerator.is_main_process:
    wandb.init(
        project=f"ddd-{task}",
        id=f"{experiment_no}",
        name=f"{experiment_no}",
        resume="allow",
        config=config,
        save_code=True

    )

    artifact = wandb.Artifact('train_notebook', type='code')
    artifact.add_file('./train.py')
    wandb.log_artifact(artifact)

optimizer.zero_grad(set_to_none=True)

for epoch in range(start_epoch, config["epochs"]):
    avg_train_loss, train_results = run_train_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        accelerator=accelerator,
        criterion=criterion,
        frames_per_window=config["frames_per_window"],
        classification_type=config["classification_type"],
        epoch=epoch,
        epochs=config["epochs"],
    )


    avg_val_loss, val_results = run_validation(
        model=model,
        val_loader=val_loader,
        accelerator=accelerator,
        criterion=criterion,
        frames_per_window=config["frames_per_window"],
        classification_type=config["classification_type"],
        id_to_group=val_id_to_group,
        epoch=epoch,
        group_stats_dir="validation_group_stats",
    )


    val_loader.reset()

    if accelerator.is_main_process:
        log_epoch_results(
            epoch=epoch,
            avg_train_loss=avg_train_loss,
            train_results=train_results,
            avg_val_loss=avg_val_loss,
            val_results=val_results,
            log_file=log_file,
            accelerator=accelerator,
            classification_type=config["classification_type"],
        )

        best_val_loss, best_val_accuracy = save_best_models(
            model=model,
            accelerator=accelerator,
            epoch=epoch,
            avg_val_loss=avg_val_loss,
            val_results=val_results,
            best_val_loss=best_val_loss,
            best_val_accuracy=best_val_accuracy,
            best_loss_model_path=best_loss_model_path,
            best_accuracy_model_path=best_accuracy_model_path,
        )

    accelerator.wait_for_everyone()
    accelerator.save_state(checkpoint_dir, safe_serialization=False)

    if accelerator.is_main_process:
        torch.save(
            {
                "epoch": epoch + 1,
                "best_val_loss": best_val_loss,
                "best_val_accuracy": best_val_accuracy,
                "current_best_threshold": val_results.get("best_threshold", 0.5),
                "current_best_precision": val_results.get("best_precision", val_results.get("precision")),
                "current_best_recall": val_results.get("best_recall", val_results.get("recall")),
                "current_best_f1": val_results.get("best_f1", val_results.get("f1")),
            },
            training_state_path,
        )

if accelerator.is_main_process:
    wandb.finish()



In [ ]:
!uv pip install evaluate wandb nvidia-dali-cuda120 -q
!wandb login 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
!accelerate launch --num_processes=2  train.py